# Notebook 04 — A/B Test Analysis
**Experiment:** Does a 2-step checkout (Variant B) convert better than 3-step (Variant A)?
**Method:** Two-proportion z-test with pre-calculated sample size and guardrail checks.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.insert(0, "..")
from analysis.ab_test_stats import (
    calculate_sample_size, run_ab_test,
    check_guardrail_metric, check_novelty_effect,
    generate_recommendation
)

from pathlib import Path
SAMPLE_DIR = Path("../data/sample")
sns.set_theme(style="whitegrid", font_scale=1.1)
print("Setup complete.")

## 1. Sample Size Calculation
This must be done BEFORE looking at results. Pre-registering the required
sample size prevents stopping early when results look good (p-hacking).

In [ ]:
ss = calculate_sample_size(baseline_rate=0.38, mde=0.02, alpha=0.05, power=0.80)
print("=== Pre-Experiment Sample Size Calculation ===")
for k, v in ss.items():
    print(f"  {k:35s}: {v}")
print(f"
Required {ss['sample_size_per_variant']:,} sessions per variant before stopping.")

## 2. Load Experiment Data

In [ ]:
df_events = pd.read_csv(SAMPLE_DIR / "events_sample.csv")
df_events["timestamp"] = pd.to_datetime(df_events["timestamp"])

# Build session-level dataset
sessions = df_events.groupby("session_id").agg(
    user_id      = ("user_id", "first"),
    variant      = ("variant", "first"),
    device       = ("device", "first"),
    channel      = ("channel", "first"),
    session_date = ("timestamp", lambda x: x.min().date()),
    converted    = ("event_type", lambda x: int("purchase" in x.values))
).reset_index()

print(f"Total sessions:  {len(sessions):,}")
print(f"Variant A:       {(sessions['variant']=='A').sum():,}")
print(f"Variant B:       {(sessions['variant']=='B').sum():,}")
print(f"
Conversion rates:")
for v in ['A','B']:
    rate = sessions[sessions["variant"]==v]["converted"].mean()
    print(f"  Variant {v}: {rate*100:.2f}%")
display(sessions.head())

## 3. Primary Metric — Conversion Rate Z-Test

In [ ]:
control   = sessions[sessions["variant"]=="A"]
treatment = sessions[sessions["variant"]=="B"]

results = run_ab_test(
    control_conversions   = control["converted"].sum(),
    control_sessions      = len(control),
    treatment_conversions = treatment["converted"].sum(),
    treatment_sessions    = len(treatment)
)

print("=== A/B Test Results ===")
for k, v in results.items():
    print(f"  {k:35s}: {v}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("A/B Test — Checkout Simplification", fontsize=14, fontweight="bold")

variants = ["A (Control)", "B (Treatment)"]
rates    = [results["control_rate"]*100, results["treatment_rate"]*100]
colors   = ["#2196F3", "#4CAF50"]
bars = axes[0].bar(variants, rates, color=colors, edgecolor="white", width=0.5)
for bar, rate in zip(bars, rates):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                 f"{rate:.2f}%", ha="center", fontweight="bold")
axes[0].set_ylim(0, max(rates)*1.2)
axes[0].set_title("Conversion Rate by Variant")
axes[0].set_ylabel("Conversion Rate (%)")

axes[1].barh(["Lift (pp)", "CI Lower", "CI Upper"],
             [results["lift_absolute_pp"], results["ci_lower_pp"], results["ci_upper_pp"]],
             color=["coral","lightblue","lightblue"])
axes[1].axvline(0, color="black", linestyle="--")
axes[1].set_title("Lift Estimate with 95% CI")
axes[1].set_xlabel("Percentage Points")

plt.tight_layout()
plt.savefig("../data/sample/ab_test_results.png", dpi=120, bbox_inches="tight")
plt.show()

## 4. Guardrail Metrics

In [ ]:
np.random.seed(42)
control_aov   = np.random.normal(55, 20, len(control))
treatment_aov = np.random.normal(54, 20, len(treatment))

guardrail_aov = check_guardrail_metric(control_aov, treatment_aov, "avg_order_value")
print("=== Guardrail: Average Order Value ===")
for k, v in guardrail_aov.items():
    print(f"  {k:35s}: {v}")

## 5. Novelty Effect Check

In [ ]:
novelty = check_novelty_effect(sessions, date_col="session_date")
print("=== Novelty Effect Check ===")
for k, v in novelty.items():
    print(f"  {k:35s}: {v}")

## 6. Segment Analysis

In [ ]:
print("=== Segment Breakdown ===")
for segment_col in ["device", "channel"]:
    print(f"
By {segment_col}:")
    seg = sessions.groupby([segment_col, "variant"])["converted"].agg(["sum","count"]).reset_index()
    seg["conversion_rate"] = (seg["sum"]/seg["count"]*100).round(2)
    pivot = seg.pivot_table(index=segment_col, columns="variant", values="conversion_rate")
    pivot["lift_pp"] = pivot["B"] - pivot["A"]
    display(pivot.round(2))

## 7. Final Recommendation

In [ ]:
rec = generate_recommendation(results, [guardrail_aov], ss)
print(rec)